In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb
!unzip -o -q /content/drive/MyDrive/ledgerly/ledgerly_data.zip -d /content/drive/MyDrive/ledgerly/raw
!ls -lh /content/drive/MyDrive/ledgerly/raw

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
unzip:  cannot find or open /content/drive/MyDrive/ledgerly/ledgerly_data.zip, /content/drive/MyDrive/ledgerly/ledgerly_data.zip.zip or /content/drive/MyDrive/ledgerly/ledgerly_data.zip.ZIP.
ls: cannot access '/content/drive/MyDrive/ledgerly/raw': No such file or directory


In [ ]:
import duckdb

RAW = '/content/drive/MyDrive/ledgerly_data'
con = duckdb.connect('/content/ledgerly.duckdb')

def q(sql):
    return con.sql(sql).df()

tables = ['touchpoints', 'leads', 'accounts', 'account_revenue_monthly', 'marketing_spend_daily']
for t in tables:
    con.sql(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv('{RAW}/{t}.csv', header=true)")
    print(t, con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

touchpoints 1811339
leads 181711
accounts 40812
account_revenue_monthly 282051
marketing_spend_daily 11660


In [ ]:
q("""
SELECT 'touchpoints' AS tbl, COUNT(*) AS n_rows, MIN(event_ts)::DATE AS min_date, MAX(event_ts)::DATE AS max_date FROM touchpoints
UNION ALL SELECT 'leads', COUNT(*), MIN(signup_ts)::DATE, MAX(signup_ts)::DATE FROM leads
UNION ALL SELECT 'accounts', COUNT(*), MIN(funded_date), MAX(funded_date) FROM accounts
UNION ALL SELECT 'account_revenue_monthly', COUNT(*), MIN(revenue_month), MAX(revenue_month) FROM account_revenue_monthly
UNION ALL SELECT 'marketing_spend_daily', COUNT(*), MIN(spend_date), MAX(spend_date) FROM marketing_spend_daily
""")

# %% Touchpoints run Nov 2023 – Mar 2026.	Some touches fall outside the 2024-25 window and must be filtered

,tbl,n_rows,min_date,max_date
0,touchpoints,1811339,2023-11-03,2026-03-31
1,leads,181711,2024-01-01,2025-12-31
2,accounts,40812,2024-01-04,2026-01-20
3,account_revenue_monthly,282051,2024-01-01,2025-12-01
4,marketing_spend_daily,11660,2024-01-01,2025-12-31


In [ ]:
# %% Summary: untagged vs tagged
q("""
SELECT CASE WHEN utm_source IS NULL AND utm_medium IS NULL THEN 'untagged (NULL)' ELSE 'tagged' END AS utm_status,
 COUNT(*) AS n,
 ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM touchpoints
GROUP BY 1
UNION ALL
SELECT 'TOTAL', COUNT(*), 100.0 FROM touchpoints
""")

,utm_status,n,pct
0,tagged,1601573,88.4
1,untagged (NULL),209766,11.6
2,TOTAL,1811339,100.0


In [ ]:
# %% Tagged rows only: every source + medium combination
q("""
SELECT utm_source, utm_medium, COUNT(*) AS n
FROM touchpoints
WHERE utm_source IS NOT NULL OR utm_medium IS NOT NULL
GROUP BY utm_source, utm_medium
ORDER BY n DESC
""")

,utm_source,utm_medium,n
0,google,paid_search,24917
1,Google,CPC,24799
2,google_ads,paid_search,24768
3,bing,paid_search,24731
4,google,CPC,24716
...,...,...,...
100,accountant_portal,referral,5878
101,referral,Referral,5817
102,Partner,referral,5796
103,partner,referral,5789


In [ ]:
q("""
SELECT channel, currency, COUNT(*) AS n_rows, ROUND(SUM(spend_amount)) AS spend
FROM marketing_spend_daily
GROUP BY channel, currency
ORDER BY channel
""")

,channel,currency,n_rows,spend
0,Accountant Partners,USD,503,1184402.0
1,Content/SEO,USD,704,1559594.0
2,Events & Webinars,USD,488,1654564.0
3,G2/Capterra,USD,744,2793888.0
4,Google Ads,USD,654,5198003.0
5,Google Ads,CAD,78,431362.0
6,LinkedIn,CAD,100,252666.0
7,LinkedIn,USD,634,3164936.0
8,LinkedIn Ads,USD,616,3167084.0
9,LinkedIn Ads,CAD,86,202450.0


In [ ]:
q("""
SELECT company_size, COUNT(*) AS n
FROM leads
GROUP BY company_size
ORDER BY n DESC
""")

,company_size,n
0,<10,23602
1,1 to 10,23577
2,1-10,23285
3,1-10 employees,23156
4,11-50,20175
5,11-50 employees,20144
6,11 to 50,19958
7,None,7258
8,50+,5209
9,51-200 employees,5166


In [ ]:
q("""
SELECT COUNT(*) AS test_leads
FROM leads
WHERE LOWER(email) LIKE '%@ledgerly.com' OR company_name LIKE 'TEST%'
""")

,test_leads
0,600


In [ ]:
q("""
SELECT COUNT(*) AS duplicate_emails, SUM(n) - COUNT(*) AS extra_rows
FROM (
 SELECT LOWER(email) AS email, COUNT(*) AS n
 FROM leads
 GROUP BY LOWER(email)
HAVING COUNT(*) > 1
)
""")

,duplicate_emails,extra_rows
0,5111,5111.0


In [ ]:
q("""
WITH d AS (
 SELECT DISTINCT lead_id, event_ts, utm_source, utm_medium, utm_campaign
 FROM touchpoints
)
SELECT (SELECT COUNT(*) FROM touchpoints) AS total_rows,
 (SELECT COUNT(*) FROM d) AS distinct_rows,
 (SELECT COUNT(*) FROM touchpoints) - (SELECT COUNT(*) FROM d) AS duplicates
""")

,total_rows,distinct_rows,duplicates
0,1811339,1775999,35340


In [ ]:
# %% lead_id not in leads
q("""
SELECT COUNT(*) AS orphan_touches
FROM touchpoints t
LEFT JOIN leads l ON l.lead_id = t.lead_id
WHERE l.lead_id IS NULL
""")

,orphan_touches
0,9000


In [ ]:
q("""
SELECT COUNT(*) AS negative_rows, ROUND(SUM(interchange_revenue_usd)) AS negative_total
FROM account_revenue_monthly
WHERE interchange_revenue_usd < 0
""")


,negative_rows,negative_total
0,4246,-670035.0


In [ ]:
q("""
SELECT revenue_month,
 COUNT(*) AS n_rows,
 COUNT(DISTINCT account_id) AS n_accounts,
 COUNT(*) - COUNT(DISTINCT account_id) AS extra_rows
FROM account_revenue_monthly
WHERE revenue_month >= DATE '2025-01-01'
GROUP BY revenue_month
ORDER BY revenue_month
""")


,revenue_month,n_rows,n_accounts,extra_rows
0,2025-01-01,12092,12092,0
1,2025-02-01,13130,13130,0
2,2025-03-01,14234,14234,0
3,2025-04-01,15201,15201,0
4,2025-05-01,16095,16095,0
5,2025-06-01,23492,16780,6712
6,2025-07-01,17522,17522,0
7,2025-08-01,18274,18274,0
8,2025-09-01,19144,19144,0
9,2025-10-01,20059,20059,0


In [ ]:
q("""
WITH all_days AS (
 SELECT d::DATE AS spend_date
 FROM generate_series(DATE '2024-01-01', DATE '2025-12-31', INTERVAL 1 DAY) AS g(d)
)
SELECT MIN(spend_date) AS first_missing, MAX(spend_date) AS last_missing, COUNT(*) AS days_missing
FROM all_days
WHERE spend_date NOT IN (
 SELECT spend_date FROM marketing_spend_daily WHERE campaign LIKE 'li_%'
)
""")

,first_missing,last_missing,days_missing
0,2025-03-03,2025-03-14,12


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE clean_leads AS
WITH ranked AS (
 SELECT *,
   ROW_NUMBER() OVER (PARTITION BY LOWER(email) ORDER BY signup_ts) AS rn
 FROM leads
 WHERE LOWER(email) NOT LIKE '%@ledgerly.com' AND company_name NOT LIKE 'TEST%'
)
SELECT lead_id,
 LOWER(email) AS email,
 company_name,
 industry,
 signup_ts,
 CASE WHEN company_size LIKE '1-10%' OR company_size IN ('1 to 10', '<10') THEN '1-10'
      WHEN company_size LIKE '11%' THEN '11-50'
      WHEN company_size LIKE '51%' OR company_size = '50+' THEN '51-200' -- assumption: 50+ = 51-200
 ELSE 'unknown' END AS company_size
FROM ranked
WHERE rn = 1
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE clean_touchpoints AS
WITH dedup AS (
 SELECT DISTINCT lead_id, event_ts, utm_source, utm_medium, utm_campaign
 FROM touchpoints
),
mapped AS (
 SELECT lead_id, event_ts, utm_campaign,
  CASE
   WHEN utm_source IS NULL AND utm_medium IS NULL THEN 'direct'
   WHEN LOWER(utm_source) IN ('linkedin', 'li', 'lnkd') THEN 'linkedin' -- source first:
   WHEN LOWER(utm_source) IN ('g2', 'capterra', 'getapp') THEN 'review_sites'  -- linkedin/capterra
   WHEN LOWER(utm_source) IN ('partner', 'accountant_portal', 'referral') THEN 'partner'  -- also use 'cpc'
   WHEN LOWER(utm_source) IN ('webinar', 'on24', 'zoom') THEN 'webinar'
   WHEN LOWER(utm_source) IN ('hubspot', 'email') THEN 'email'
   WHEN LOWER(utm_medium) IN ('cpc', 'ppc', 'paid_search') THEN 'paid_search' -- then medium:
   WHEN LOWER(utm_medium) IN ('organic', 'seo', 'content') THEN 'content_seo' -- google paid vs organic
   ELSE 'other'
  END AS channel
  FROM dedup
)
SELECT m.*
FROM mapped m
JOIN clean_leads l ON l.lead_id = m.lead_id -- drops orphans, test + duplicate leads
WHERE m.event_ts <= l.signup_ts -- after signup = can't have caused it AND m.event_ts >= l.signup_ts - INTERVAL 90 DAY -- 90-day lookback window
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE clean_spend AS
SELECT spend_date,
 CASE
  WHEN channel IN ('Paid Search', 'paid_search', 'Google Ads') THEN 'paid_search'
  WHEN channel IN ('LinkedIn', 'linkedin_ads', 'LinkedIn Ads') THEN 'linkedin'
  WHEN channel IN ('Review Sites', 'review_sites', 'G2/Capterra') THEN 'review_sites'
  WHEN channel IN ('Webinars', 'webinar', 'Events & Webinars') THEN 'webinar'
  WHEN channel IN ('Partner Program', 'partner', 'Accountant Partners') THEN 'partner'
  WHEN channel IN ('Content/SEO', 'content', 'SEO') THEN 'content_seo'
 END AS channel,
  campaign,
  CASE WHEN currency = 'CAD' THEN spend_amount * 0.73 ELSE spend_amount END AS spend_usd
FROM marketing_spend_daily
""")


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE clean_revenue AS
SELECT DISTINCT account_id, revenue_month,
 card_volume_usd, interchange_revenue_usd, subscription_revenue_usd,
 interchange_revenue_usd + subscription_revenue_usd AS total_revenue_usd
FROM account_revenue_monthly
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE clean_accounts AS
SELECT a.account_id, a.lead_id, a.funded_date, a.plan_tier, a.account_status, a.closed_date
FROM accounts a
JOIN clean_leads l ON l.lead_id = a.lead_id
WHERE a.funded_date IS NOT NULL
""")


In [ ]:
q("""
SELECT 'leads' AS tbl, (SELECT COUNT(*) FROM leads) AS raw_rows, (SELECT COUNT(*) FROM clean_leads) AS clean_rows
UNION ALL SELECT 'touchpoints', (SELECT COUNT(*) FROM touchpoints), (SELECT COUNT(*) FROM clean_touchpoints)
UNION ALL SELECT 'accounts (funded only)', (SELECT COUNT(*) FROM accounts), (SELECT COUNT(*) FROM clean_accounts)
UNION ALL SELECT 'revenue', (SELECT COUNT(*) FROM account_revenue_monthly), (SELECT COUNT(*) FROM clean_revenue)
UNION ALL SELECT 'spend', (SELECT COUNT(*) FROM marketing_spend_daily), (SELECT COUNT(*) FROM clean_spend)
""")


,tbl,raw_rows,clean_rows
0,leads,181711,176000
1,touchpoints,1811339,1485540
2,accounts (funded only),40812,30289
3,revenue,282051,275339
4,spend,11660,11660


In [ ]:
q("""
SELECT (SELECT COUNT(*) FROM clean_touchpoints WHERE channel = 'other')  AS unmapped_touches,
 (SELECT COUNT(*) FROM clean_spend WHERE channel IS NULL) AS unmapped_spend,
 (SELECT COUNT(*) FROM clean_leads WHERE company_size = 'unknown') AS unknown_size
""")

,unmapped_touches,unmapped_spend,unknown_size
0,0,0,7020


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE lead_journeys AS
SELECT lead_id, channel, utm_campaign, event_ts,
 ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY event_ts) AS touch_n, -- 1 = first touch
 ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY event_ts DESC) AS touch_n_rev, -- 1 = last touch
 COUNT(*) OVER (PARTITION BY lead_id) AS n_touches
FROM clean_touchpoints
""")


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE attribution AS
SELECT j.channel,
 SUM(CASE WHEN j.touch_n_rev = 1 THEN 1 ELSE 0 END) AS last_touch,
 SUM(CASE WHEN j.touch_n = 1 THEN 1 ELSE 0 END) AS first_touch,
 ROUND(SUM(1.0 / j.n_touches), 0) AS linear -- each touch gets 1/n of the credit
FROM lead_journeys j
JOIN clean_accounts a ON a.lead_id = j.lead_id
GROUP BY j.channel
""")


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE paid_search_last_touch AS
SELECT j.utm_campaign, COUNT(*) AS last_touch_accounts
FROM lead_journeys j
JOIN clean_accounts a ON a.lead_id = j.lead_id
WHERE j.touch_n_rev = 1 AND j.channel = 'paid_search'
GROUP BY j.utm_campaign
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE funnel_by_channel AS
WITH first_touch AS (
 SELECT lead_id, channel FROM lead_journeys WHERE touch_n = 1
)
SELECT f.channel,
 COUNT(*) AS leads,
 COUNT(a.account_id) AS funded,
 ROUND(100.0 * COUNT(a.account_id) / COUNT(*), 1) AS funded_pct,
 ROUND(100.0 * SUM(CASE WHEN a.closed_date < a.funded_date + INTERVAL 12 MONTH THEN 1 ELSE 0 END) / NULLIF(COUNT(a.account_id), 0), 1) AS churn_12m_pct
FROM first_touch f
JOIN clean_leads l ON l.lead_id = f.lead_id
LEFT JOIN clean_accounts a ON a.lead_id = f.lead_id
WHERE l.signup_ts < TIMESTAMP '2025-01-01'
GROUP BY f.channel
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE account_12m AS
SELECT a.account_id, a.lead_id, COALESCE(SUM(r.total_revenue_usd), 0) AS rev_12m
FROM clean_accounts a
JOIN clean_leads l ON l.lead_id = a.lead_id
LEFT JOIN clean_revenue r
  ON r.account_id = a.account_id
 AND r.revenue_month < DATE_TRUNC('month', a.funded_date) + INTERVAL 12 MONTH
WHERE l.signup_ts < TIMESTAMP '2025-01-01'
GROUP BY a.account_id, a.lead_id
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE channel_economics AS
WITH rev AS (
 SELECT j.channel,
  SUM(1.0 / j.n_touches) AS accounts_credited,
  SUM(a.rev_12m / j.n_touches) AS rev_12m
 FROM lead_journeys j
 JOIN account_12m a ON a.lead_id = j.lead_id
 GROUP BY j.channel
),
sp AS (
 SELECT channel, SUM(spend_usd) AS spend_2024
 FROM clean_spend
 WHERE spend_date < DATE '2025-01-01'
 GROUP BY channel
)
SELECT r.channel,
 ROUND(sp.spend_2024) AS spend_2024,
 ROUND(r.accounts_credited) AS accounts_credited,
 ROUND(sp.spend_2024 / r.accounts_credited) AS cac,
 ROUND(r.rev_12m) AS rev_12m,
 ROUND(r.rev_12m / sp.spend_2024, 2) AS rev_per_dollar
FROM rev r
LEFT JOIN sp ON sp.channel = r.channel
""")

In [ ]:
q("SELECT * FROM attribution ORDER BY last_touch DESC")

,channel,last_touch,first_touch,linear
0,paid_search,17037.0,5136.0,8399.0
1,linkedin,2817.0,6152.0,4959.0
2,content_seo,2776.0,4522.0,4630.0
3,direct,2369.0,3428.0,3049.0
4,review_sites,2295.0,2713.0,3737.0
5,webinar,1642.0,3656.0,2982.0
6,partner,1353.0,4682.0,2533.0


In [ ]:
q("SELECT * FROM paid_search_last_touch ORDER BY last_touch_accounts DESC")

,utm_campaign,last_touch_accounts
0,brand_ledgerly,14296
1,nonbrand_expense_cards,1418
2,nonbrand_ap_automation,1323


In [ ]:
q("SELECT * FROM funnel_by_channel ORDER BY funded_pct DESC")

,channel,leads,funded,funded_pct,churn_12m_pct
0,partner,4420,2074,46.9,16.3
1,webinar,5941,1667,28.1,25.7
2,linkedin,11100,2728,24.6,22.9
3,direct,9113,1515,16.6,28.8
4,paid_search,14679,2283,15.6,34.8
5,content_seo,14198,1979,13.9,33.1
6,review_sites,18350,1245,6.8,59.8


In [ ]:
q("SELECT * FROM channel_economics ORDER BY rev_per_dollar DESC NULLS LAST")

,channel,spend_2024,accounts_credited,cac,rev_12m,rev_per_dollar
0,partner,1520950.0,1124.0,1353.0,8503625.0,5.59
1,content_seo,2172466.0,2046.0,1062.0,8873328.0,4.08
2,webinar,2138340.0,1339.0,1597.0,7119018.0,3.33
3,linkedin,4558152.0,2209.0,2064.0,14107661.0,3.10
4,paid_search,7541088.0,3736.0,2018.0,17084474.0,2.27
5,review_sites,3690543.0,1680.0,2197.0,6693203.0,1.81
6,direct,NaN,1357.0,NaN,6384615.0,NaN


In [ ]:
q("SELECT SUM(last_touch), SUM(first_touch), SUM(linear) FROM attribution")

,sum(last_touch),sum(first_touch),sum(linear)
0,30289.0,30289.0,30289.0


In [ ]:
# %% Reallocation scenario: move ~$2.6M from weakest to strongest channels
con.sql("""
CREATE OR REPLACE TABLE scenario AS
WITH shift AS ( -- % change in each channel's 2024 budget
  SELECT * FROM (VALUES
   ('review_sites', -0.30),
   ('paid_search',  -0.20),
   ('partner',       0.70),
   ('webinar',       0.40),
   ('content_seo',   0.35)
  ) AS s(channel, pct_change)
),
effect AS ( -- how well NEW money works vs today's average dollar
  SELECT * FROM (VALUES (0.5), (0.75), (1.0)) AS e(effectiveness)
)
SELECT e.effectiveness,
 ROUND(SUM(c.spend_2024 * s.pct_change)) AS net_spend_change,
 ROUND(SUM(CASE WHEN s.pct_change < 0 THEN -c.spend_2024 * s.pct_change ELSE 0 END)) AS dollars_moved,
 ROUND(SUM(c.spend_2024 * s.pct_change * c.rev_per_dollar * CASE WHEN s.pct_change > 0 THEN e.effectiveness ELSE 1 END)) AS rev_12m_change
FROM channel_economics c
JOIN shift s ON s.channel = c.channel
CROSS JOIN effect e
GROUP BY e.effectiveness
ORDER BY e.effectiveness
""")
q("SELECT * FROM scenario")

,effectiveness,net_spend_change,dollars_moved,rev_12m_change
0,0.50,64984.0,2615381.0,523395.0
1,0.75,64984.0,2615381.0,3498902.0
2,1.00,64984.0,2615381.0,6474409.0


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE monthly_channel AS
WITH sp AS (
 SELECT DATE_TRUNC('month', spend_date) AS month, channel, SUM(spend_usd) AS spend_usd
 FROM clean_spend
 GROUP BY 1, 2
),
ld AS (
 SELECT DATE_TRUNC('month', l.signup_ts)::DATE AS month, j.channel,
   COUNT(*) AS leads, COUNT(a.account_id) AS funded_accounts
 FROM lead_journeys j
 JOIN clean_leads l ON l.lead_id = j.lead_id
 LEFT JOIN clean_accounts a ON a.lead_id = j.lead_id
 WHERE j.touch_n = 1
 GROUP BY 1, 2
)
SELECT COALESCE(ld.month, sp.month) AS month,
 COALESCE(ld.channel, sp.channel) AS channel,
 ROUND(COALESCE(sp.spend_usd, 0), 2) AS spend_usd,
 COALESCE(ld.leads, 0) AS first_touch_leads,
 COALESCE(ld.funded_accounts, 0) AS funded_accounts
FROM ld
FULL OUTER JOIN sp ON sp.month = ld.month AND sp.channel = ld.channel
""")
q("SELECT COUNT(*) AS n_rows, SUM(first_touch_leads) AS leads, SUM(funded_accounts) AS funded FROM monthly_channel")

,n_rows,leads,funded
0,168,176000.0,30289.0


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE cleaning_impact AS
SELECT 'Total revenue (USD)' AS metric,
 ROUND((SELECT SUM(interchange_revenue_usd + subscription_revenue_usd) FROM account_revenue_monthly)) AS raw,
 ROUND((SELECT SUM(total_revenue_usd) FROM clean_revenue)) AS clean
UNION ALL SELECT 'Total spend (USD)',
 ROUND((SELECT SUM(spend_amount) FROM marketing_spend_daily)),
 ROUND((SELECT SUM(spend_usd) FROM clean_spend))
UNION ALL SELECT 'Leads', (SELECT COUNT(*) FROM leads), (SELECT COUNT(*) FROM clean_leads)
UNION ALL SELECT 'Touchpoints', (SELECT COUNT(*) FROM touchpoints), (SELECT COUNT(*) FROM clean_touchpoints)
""")
q("SELECT *, ROUND(100.0 * (raw - clean) / raw, 1) AS pct_overstated FROM cleaning_impact")

,metric,raw,clean,pct_overstated
0,Total revenue (USD),141382795.0,137981997.0,2.4
1,Total spend (USD),49277382.0,48730921.0,1.1
2,Leads,181711.0,176000.0,3.1
3,Touchpoints,1811339.0,1485540.0,18.0


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE model_comparison AS
WITH credit AS (
 SELECT j.channel,
  SUM(CASE WHEN j.touch_n_rev = 1 THEN a.rev_12m ELSE 0 END) AS rev_last,
  SUM(CASE WHEN j.touch_n = 1     THEN a.rev_12m ELSE 0 END) AS rev_first,
  SUM(a.rev_12m / j.n_touches) AS rev_linear
 FROM lead_journeys j
 JOIN account_12m a ON a.lead_id = j.lead_id
 GROUP BY j.channel
)
SELECT c.channel,
 ROUND(c.rev_last   / e.spend_2024, 2) AS rpd_last_touch,
 ROUND(c.rev_first  / e.spend_2024, 2) AS rpd_first_touch,
 ROUND(c.rev_linear / e.spend_2024, 2) AS rpd_linear
FROM credit c
JOIN channel_economics e ON e.channel = c.channel
WHERE e.spend_2024 IS NOT NULL
""")
q("SELECT * FROM model_comparison ORDER BY rpd_linear DESC")

,channel,rpd_last_touch,rpd_first_touch,rpd_linear
0,partner,2.70,11.64,5.59
1,content_seo,2.76,2.73,4.08
2,webinar,1.86,4.25,3.33
3,linkedin,1.65,4.55,3.10
4,paid_search,5.03,0.97,2.27
5,review_sites,1.21,0.47,1.81


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE segment_mix AS
WITH ft AS (
 SELECT lead_id, channel FROM lead_journeys WHERE touch_n = 1
)
SELECT ft.channel,
 COUNT(*) AS accounts,
 ROUND(100.0 * AVG(CASE WHEN l.company_size = '1-10'   THEN 1 ELSE 0 END), 1) AS pct_1_10,
 ROUND(100.0 * AVG(CASE WHEN l.company_size = '51-200' THEN 1 ELSE 0 END), 1) AS pct_51_200,
 ROUND(AVG(a.rev_12m)) AS avg_rev_12m
FROM account_12m a
JOIN ft ON ft.lead_id = a.lead_id
JOIN clean_leads l ON l.lead_id  = a.lead_id
GROUP BY ft.channel
""")
q("SELECT * FROM segment_mix ORDER BY avg_rev_12m DESC")

,channel,accounts,pct_1_10,pct_51_200,avg_rev_12m
0,partner,2074,26.8,24.0,8538.0
1,linkedin,2728,23.2,29.2,7605.0
2,webinar,1667,31.1,21.0,5450.0
3,direct,1515,42.8,13.9,4140.0
4,paid_search,2283,46.1,11.4,3188.0
5,content_seo,1979,50.8,10.5,2998.0
6,review_sites,1245,61.1,6.0,1399.0


In [ ]:
q("""
SELECT l.company_size, COUNT(*) AS accounts, ROUND(AVG(a.rev_12m)) AS avg_rev_12m
FROM account_12m a JOIN clean_leads l ON l.lead_id = a.lead_id
GROUP BY 1 ORDER BY avg_rev_12m DESC
""")

,company_size,accounts,avg_rev_12m
0,51-200,2399,15922.0
1,unknown,544,5622.0
2,11-50,5373,4325.0
3,1-10,5175,826.0


In [ ]:
import os
OUT = '/content/drive/MyDrive/ledgerly_data/outputs'
os.makedirs(OUT, exist_ok=True)

tables = ['attribution', 'paid_search_last_touch', 'funnel_by_channel', 'channel_economics', 'scenario', 'monthly_channel', 'cleaning_impact', 'model_comparison', 'segment_mix']
for t in tables:
 con.sql(f"COPY {t} TO '{OUT}/{t}.csv' (HEADER)")
print(sorted(os.listdir(OUT)))

['attribution.csv', 'channel_economics.csv', 'cleaning_impact.csv', 'funnel_by_channel.csv', 'model_comparison.csv', 'monthly_channel.csv', 'paid_search_last_touch.csv', 'scenario.csv', 'segment_mix.csv']
